In [1]:
import pandas as pd
import json
import os
import requests

from shapely.geometry import LineString
from shapely.ops import unary_union

<a id="Route-analyser"></a>
<h3 style="color:#1f77b4;">Route Analyser</h3>
<a id="Loading-Input-URLS"></a>
<h5 style="color:#1fb4a7;">Load route addresses</h5>

In [2]:
HOME_LAT = -31.863708969515947
HOME_LON = 116.04151607642865

ROUTE_CSV = r"C:\Users\thoma\Documents\balga_r_deadwalk.csv"

streets_to_find = [
    "Stedham Way",
    "Brede Place",
    "St Kilda Road",
    "Fitzroy Place",
    "Mentone Road",
    "Grinstead Way",
]

##### Get street geometry from Overpass

In [3]:
cache_file = r"C:\Users\thoma\Documents\balga_streets.json"

# Reuse street data to prevent requesting many times
if os.path.exists(cache_file):
    with open(cache_file) as file:
        data = json.load(file)

else:
    # Overpass query for the selected Balga streets
    # NOTES FOR FUTURE:
    # [out:json] returns the result as JSON
    # area finds the Balga boundary and stores it as .a
    # way(area.a) searches for roads inside that boundary
    # ["highway"] keeps only road features
    # ["name"~"..."] matches any street name joined with |
    # out geom returns each road with its coordinates
    query = f"""
    [out:json];
    area["name"="Balga"]["admin_level"]->.a;
    way(area.a)["highway"]["name"~"{"|".join(streets_to_find)}"];
    out geom;
    """

    # Send Overpass API request
    response = requests.get(
        "https://overpass-api.de/api/interpreter",
        params={"data": query},
        timeout=100,
    )

    # Convert to JSON
    data = response.json()

    with open(cache_file, "w") as file:
        json.dump(data, file)

# Show number of returned elements
print(f"{len(data['elements'])} elements")

7 elements


##### Build road segments

In [4]:
# Each Overpass element is a map feature such as a road way
# A way contains tags such as the road name and geometry with latitude and longitude points
# Each way usually represents one section of a road rather than the entire street
road_segments = []

for element in data["elements"]:
    # Keep only road way elements
    if element["type"] != "way":
        continue

    name = element.get("tags", {}).get("name", "unknown")

    # Extract the points that trace the road segment
    coords = [
        (node["lat"], node["lon"])
        for node in element["geometry"]
    ]

    # Store the road segment and its endpoints
    road_segments.append(
        {
            "name": name,
            "coords": coords,
            "start": coords[0],
            "end": coords[-1],
        }
    )

    print(f"{name}: {len(coords)} geometry points")

print(f"\nCreated {len(road_segments)} road segments")

Mentone Road: 7 geometry points
Stedham Way: 7 geometry points
Brede Place: 2 geometry points
St Kilda Road: 10 geometry points
Grinstead Way: 6 geometry points
Fitzroy Place: 4 geometry points
Grinstead Way: 3 geometry points

Created 7 road segments


##### Build street outline 

In [5]:
# Convert each road segment into a line using shapely
# Shapely uses coordinates in longitude latitude order
street_lines = []

for segment in road_segments:
    coordinates = [
        (lon, lat)
        for lat, lon in segment["coords"]
    ]

    # A line needs at least two coordinate points
    if len(coordinates) >= 2:
        street_lines.append(LineString(coordinates))

# Merge all road lines into one geometry
all_streets = unary_union(street_lines)

# Add space around the roads to create one connected street shape
buffered_streets = all_streets.buffer(0.00025)

# Extract the outside boundary of the buffered streets
outline = buffered_streets.exterior
outline_points = list(outline.coords)

# Convert coordinates back to latitude longitude order
outline_path = [
    (lat, lon)
    for lon, lat in outline_points
]

print(f"Created outline with {len(outline_points)} points")

Created outline with 361 points
